# Unsupervised exploration (_Mushroom_)

In [1]:
from experiments.utils.constants import RANDOM_SEED, SOM_LEARNING_RATE_DECAY_FN, SOM_FIT_METHOD

VERBOSE = True
DATASET_NAME = "Mushroom"
DATASET_ID = 73
MODELING_MODE = False
MODEL_REGISTRATION_MODE = False
EXPORT_MODE = False
EXPORT_DIR = "_exports"

## Dataset

In [2]:
# fetch dataset
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=DATASET_ID)
X = dataset.data.features.values
y = dataset.data.targets.values.ravel()
print(f"Dataset shape: {X.shape}, {y.shape}")

Dataset shape: (8124, 22), (8124,)


In [3]:
# encode data
import pandas as pd
X = pd.get_dummies(pd.DataFrame(X), dummy_na=True).values

In [4]:
# scale data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
del X

## Modeling

In [5]:
from minisom_representation import calc_som_hyparams, SomRepresentation, plot_som_convergence_over_epochs

In [6]:
# use helper methods to get SOM hyperparameter recommendations
recommended_params = calc_som_hyparams(X_scaled, initial_sigma_factor=3.0)
print("Recommended SOM parameters:", recommended_params)

Recommended SOM parameters: {'d1': 21, 'd2': 22, 'sigma': 7.33}


In [7]:
# define actual hyperparameters
d1, d2, sigma = map(recommended_params.get, ("d1", "d2", "sigma"))
decay_function = SOM_LEARNING_RATE_DECAY_FN
epoch = None

In [8]:
# test candidate values for `num_iteration` hyperparameter
if MODELING_MODE:
    fig, errors_qe, errors_te = plot_som_convergence_over_epochs(
        SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=False, decay_function=decay_function),
        X_scaled,
        fit_type=SOM_FIT_METHOD, te_ceiling=.1,
        epoch_step_from=2, epoch_step_to=10, epoch_step=1,
        figsize=(16, 5), show_fig=True, verbose=VERBOSE
    )
    print(f"\nQE (first -> last): \t {errors_qe[0]:.2f} -> {errors_qe[-1]:.2f}")
    print(f"TE (first -> last): \t {errors_te[0]:.2f} -> {errors_te[-1]:.2f}")
else: print("Skipping model candidate parameters evaluation.")

Skipping model candidate parameters evaluation.


In [9]:
# test candidate values for `num_iteration` hyperparameter
if MODELING_MODE:
    fig, errors_qe, errors_te = plot_som_convergence_over_epochs(
        SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=False, decay_function=decay_function),
        X_scaled,
        fit_type=SOM_FIT_METHOD, te_ceiling=.1,
        epoch_step_from=10, epoch_step_to=20, epoch_step=2,
        figsize=(16, 5), show_fig=True, verbose=VERBOSE
    )
    print(f"\nQE (first -> last): \t {errors_qe[0]:.2f} -> {errors_qe[-1]:.2f}")
    print(f"TE (first -> last): \t {errors_te[0]:.2f} -> {errors_te[-1]:.2f}")
else: print("Skipping model candidate parameters evaluation.")

Skipping model candidate parameters evaluation.


In [10]:
# test candidate values for `num_iteration` hyperparameter
if MODELING_MODE:
    fig, errors_qe, errors_te = plot_som_convergence_over_epochs(
        SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=False, decay_function=decay_function),
        X_scaled,
        fit_type=SOM_FIT_METHOD, te_ceiling=.1,
        epoch_step_from=20, epoch_step_to=50, epoch_step=6,
        figsize=(16, 5), show_fig=True, verbose=VERBOSE
    )
    print(f"\nQE (first -> last): \t {errors_qe[0]:.2f} -> {errors_qe[-1]:.2f}")
    print(f"TE (first -> last): \t {errors_te[0]:.2f} -> {errors_te[-1]:.2f}")
else: print("Skipping model candidate parameters evaluation.")

Skipping model candidate parameters evaluation.


In [11]:
# set selected `num_iteration` as epoch
epoch = 20

In [12]:
# fit SOM representation
som_rep = SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=VERBOSE, decay_function=decay_function) \
    .fit_online(X_scaled, num_iteration=epoch)

 [ 162480 / 162480 ] 100% - 0:00:00 left 
 quantization error: 6.56711584803875

 An SOM representation has been fitted as follows:
------------------------------------------------------- 

Fit strategy: online 

Hyperparameters of SOM: 

{'input_len': 138, 'x': 21, 'y': 22, 'sigma': 7.33, 'topology': 'rectangular', 'learning_rate': 0.5, 'decay_function': 'linear_decay_to_zero', 'sigma_decay_function': 'asymptotic_decay', 'neighborhood_function': 'gaussian', 'activation_distance': 'euclidean', 'random_seed': 42, 'num_iteration': 20, 'use_epochs': True, 'random_order': True, 'verbose': True} 

Quality of SOM: 

Quantization Error (QE):	6.56711584803875
Topographic Error (TE): 	0.00012309207287050715


## Inspection of the learned 2D topology

In [13]:
from utils.plotting import PlotlyHelperArgs

In [14]:
# create Basin
from lilypond import Basin
basin = Basin.from_som_representation(som_rep, random_seed=RANDOM_SEED, verbose=VERBOSE)

In [15]:
# export basin
if EXPORT_MODE:
	from utils.export import BasinWithTrainingData
	BasinWithTrainingData(dataset_name=DATASET_NAME, basin=basin, X_train=X_scaled).export("_exports/other")

In [16]:
# traditional visuals
plot_args = dict(
    **PlotlyHelperArgs.FullStretch,
    **PlotlyHelperArgs.HiddenTicks(d1=d1, d2=d2),
    font=dict(size=35),
    title="",
)
figTrad1 = basin.legacy_pond().visualize_distance_map(**plot_args, **PlotlyHelperArgs.Figsize(w=950, h=600));
figTrad2 = basin.legacy_pond().visualize_activation_map(**plot_args, **PlotlyHelperArgs.Figsize(w=850, h=600));

In [17]:
if EXPORT_MODE:
	figTrad1.write_image(EXPORT_DIR + "/02_03_lilypond_trad_01.png")
	figTrad2.write_image(EXPORT_DIR + "/02_03_lilypond_trad_02.png")

In [18]:
# lilypond visual
basin.pond() \
    .rhizome_layer() \
    .pad_layer() \
    .petal_layer() \
    .visualize(width=800, height=600);

## Detailed inspection

In [19]:
plot_args = dict(
    **PlotlyHelperArgs.Figsize(w=400, h=400),
    **PlotlyHelperArgs.FullStretch,
    **PlotlyHelperArgs.HiddenTicks(d1=d1, d2=d2),
    font=dict(size=35),
    showlegend=False
)

In [20]:
fig1 = basin.pond() \
    .pad_layer() \
    .visualize(**plot_args);

In [21]:
fig2 = basin.pond() \
    .pad_layer(gap="nogap") \
    .visualize(**plot_args);

In [22]:
fig3 = basin.pond() \
    .pad_layer(gap="nogap") \
    .petal_layer() \
    .visualize(**plot_args);

In [23]:
fig4 = basin.pond() \
    .pad_layer(gap="nogap") \
    .rhizome_layer() \
    .visualize(**plot_args);

In [24]:
fig5 = basin.pond() \
    .pad_layer(gap="nogap") \
    .rhizome_layer(violations_only=True) \
    .visualize(**plot_args);

In [25]:
if EXPORT_MODE:
	fig1.write_image(EXPORT_DIR + "/02_03_lilypond_01.png")
	fig2.write_image(EXPORT_DIR + "/02_03_lilypond_02.png")
	fig3.write_image(EXPORT_DIR + "/02_03_lilypond_03.png")
	fig4.write_image(EXPORT_DIR + "/02_03_lilypond_04.png")
	fig5.write_image(EXPORT_DIR + "/02_03_lilypond_05.png")

## Extra figures

In [26]:
plot_args = dict(
    **PlotlyHelperArgs.Figsize(w=800, h=800),
    **PlotlyHelperArgs.FullStretch,
    **PlotlyHelperArgs.HiddenTicks(d1=d1, d2=d2),
    font=dict(size=35),
    showlegend=False
)

In [27]:
figExtra1 = basin.pond() \
    .pad_layer(gap="nogap") \
    .petal_layer(max_size=60, marker_line=dict(width=6), marker_halo=dict(line=dict(width=15, color="black"))) \
    .visualize(**plot_args);

In [28]:
figExtra2 = basin.pond() \
    .pad_layer(gap="nogap", colorscale="RdYlGn_r") \
    .rhizome_layer(min_width=10, max_width=30) \
    .visualize(**plot_args);

In [29]:
if EXPORT_MODE:
	figExtra1.write_image(EXPORT_DIR + "/02_03_lilypond_extra_01.png")
	figExtra2.write_image(EXPORT_DIR + "/02_03_lilypond_extra_02.png")

---

### The below cells are not part of the experiment. They are used to register the model in Databricks and Bianor for further interactive investigation.

---

## Register representation model in Databricks

In [30]:
from dotenv import load_dotenv
load_dotenv()

True

In [31]:
CATALOG = "workspace"
SCHEMA = "lilypond_experiments"
MODEL_NAME = f"som-{DATASET_NAME.lower()}"
MODEL_PATH = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
EXPERIMENT_NAME = f"/Users/matebalogh@ophelia-rnd.dev/Bianor_LilypondExperiments_{DATASET_NAME}"
print(MODEL_PATH, EXPERIMENT_NAME)

workspace.lilypond_experiments.som-mushroom /Users/matebalogh@ophelia-rnd.dev/Bianor_LilypondExperiments_Mushroom


In [32]:
import mlflow
import pandas as pd
from typing import Any

mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

class MLflowSomModelWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model:SomRepresentation, scaler):
        self.model = model
        self.scaler = scaler
    def predict(self, context, model_input, params: dict[str, Any] | None = None):
        """First transforms the input data via scaler, then predicts the winner node of the SOM."""
        return [self.model.som.winner(x) for x in self.scaler.transform(model_input.to_numpy())]

if MODEL_REGISTRATION_MODE:
    with mlflow.start_run():
        model = MLflowSomModelWrapper(som_rep, scaler)

        mlflow.log_metric("QE", som_rep.quantization_error)
        mlflow.log_metric("TE", som_rep.topographic_error)

        mlflow.pyfunc.log_model(
            python_model=model,
            name=MODEL_NAME,
            input_example=pd.DataFrame(X_scaled[:3]),
            pip_requirements=[
                "numpy",
                "pandas",
                "scikit-learn==1.5.2",
                "mlflow",
                "minisom",
            ],
            registered_model_name=MODEL_PATH
        )
else: print("Skipping MLflow model registration.")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
/opt/homebrew/Caskroom/miniconda/base/envs/wsom-lilypond-exp/lib/python3.11/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/09/13 02:07:05 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/doc

🏃 View run selective-rook-392 at: dbc-ca44ccd5-3843.cloud.databricks.com/ml/experiments/1916796335158891/runs/bed9e7f3825e47ed902d20a6968cdf11
🧪 View experiment at: dbc-ca44ccd5-3843.cloud.databricks.com/ml/experiments/1916796335158891


In [33]:
version = 1
registered_model = f"{MODEL_NAME}/{version}"
registered_model

'som-mushroom/1'

## Register metadata in Bianor

In [34]:
from databricks.connect import DatabricksSession
spark = DatabricksSession.builder.getOrCreate()

In [35]:
from bianor_databricks_kit import BianorRecordManager
bianor_recorder = BianorRecordManager(catalog=CATALOG, schema=SCHEMA, spark=spark)

In [36]:
registered_model_location = f"{CATALOG}.{SCHEMA}.{registered_model}"
registered_model_location

'workspace.lilypond_experiments.som-mushroom/1'

In [37]:
if MODEL_REGISTRATION_MODE:
    bianor_recorder.new_representation(name=f"{DATASET_NAME} Representation", som_model_location=registered_model_location, features_location="c.s.t") # FIXME
else: print("Skipping Bianor metadata registration.")